In [ ]:
import json
import os
from collections import Counter

In [ ]:
import json
import os
from collections import defaultdict

def calculate_tabfact_accuracy(file_path, start=0, over=2024):
    if not os.path.exists(file_path):
        print(f"❌ file not exist: {file_path}")
        return None

    total_count = 0
    step_stats = defaultdict(lambda: {"correct": 0, "total": 0})

    wrong_ids_by_step = defaultdict(list)

    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            
            try:
                data = json.loads(line)
                item_id = data.get("ids", data.get("pair_id", f"line_{total_count}"))
                try:
                    num = int(item_id.split("-")[1])
                except (IndexError, ValueError):
                    num = 0
                if not (num >= start and num <= over):
                    continue
                
                total_count += 1

                raw_label = data.get("true_answer")
                label_str = "true" if str(raw_label).lower() in ["1", "true"] else "false"
            
                pred_first = str(data.get("first_result", "")).strip().lower()
                step_stats[0]["total"] += 1
                if pred_first == label_str:
                    step_stats[0]["correct"] += 1
                else:
                    wrong_ids_by_step[0].append(item_id)

                raw_final = data.get("final_result", [])
                if isinstance(raw_final, list):
                    for i, refine_pred in enumerate(raw_final):
                        step_idx = i + 1 
                        pred_str = str(refine_pred).strip().lower()
                        
                        step_stats[step_idx]["total"] += 1
                        if pred_str == label_str:
                            step_stats[step_idx]["correct"] += 1
                        else:
                            wrong_ids_by_step[step_idx].append(item_id)
                            
            except json.JSONDecodeError:
                continue

    if total_count == 0:
        print("⚠️ no data")
        return None

   
    print(f"\n{'='*50}")
    print(f"{'step':<15} | {'acc':<10} | {'True/Total':<12}")
    print(f"{'-'*50}")
    
   
    for step in sorted(step_stats.keys()):
        label = "First Result" if step == 0 else f"Refine {step}"
        correct = step_stats[step]["correct"]
        total = step_stats[0]["total"]
        acc = correct / total if total > 0 else 0
        print(f"{label:<15} | {acc:>9.10%} | {correct:>5}/{total:<5}")
    
    print(f"{'='*50}\n")
    
    return {
        "total_rows": total_count,
        "accuracy_by_step": {k: v["correct"]/v["total"] for k, v in step_stats.items()},
        "wrong_ids": dict(wrong_ids_by_step)
    }


In [ ]:
def create_path(num , model_name , type ,  route):
    return f"../result/run{num}_results/tabfact/{model_name}/tabfact-{model_name}-cols_{type}-routing_{route}/result.jsonl"


# How to eval

You can choose to directly pass in the results.josn file, or fill in the relevant configuration in the configuration file if you do not change result path in config.
The parameters for create_path are as follows:
* **num**: task_run_ids in config
* **model_name**: model_name in config
* **type**: col_select_type in config
* **route_type**: routing_strategy in config

In [ ]:
# example
path = create_path(1 , "gpt-4o-mini"  , "entity"  , "hybrid")
_= calculate_tabfact_accuracy(path)